# Translate one Steam description batch with Qwen on Google Colab

This notebook processes **one input CSV** from Google Drive and writes translated rows to **one output CSV** in `Translation Done`. It installs the required Python packages, downloads the open Qwen3-4B model from Hugging Face into the Colab runtime, loads it in 4-bit mode on the available GPU, and translates non-English portions while preserving the English text and meaning.

After each successful translation, it atomically saves the output CSV, then atomically rewrites the input CSV without that AppID. If Colab stops between those two saves, rerunning the notebook detects the AppID in the output and removes it from the input without translating it again. The original input is intentionally modified game by game, as requested. Keep the input and output in Google Drive so they survive Colab runtime deletion.

## Before running

1. In Colab, choose a GPU runtime.
2. Copy the selected `translation_batch_XX_of_09.csv` into the `Translation Needed` folder in your Google Drive.
3. Set `DATA_PREP_DIR` in the configuration cell to the corresponding `Data Prep` folder in Drive.
4. Set `BATCH_FILENAME` to the one CSV this runtime should process.
5. Run the cells in order. The model downloads once per fresh Colab runtime.
6. When the batch is complete, the translated CSV will have the same filename in `Translation Done`.

The GPU, model, and free Colab runtime are not guaranteed to be available for a particular duration. This notebook saves progress after every game, so the same input/output pair can be resumed.

In [ ]:
# Install dependencies. Colab already supplies PyTorch; these packages add Qwen3 support and 4-bit GPU loading.
%pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes "pandas==2.2.3" regex

In [ ]:
from pathlib import Path
import os
import re
import time

import pandas as pd
import torch
from google.colab import drive

drive.mount("/content/drive")

# CHANGE THIS to the Data Prep folder you created in Google Drive.
DATA_PREP_DIR = Path("/content/drive/MyDrive/Data Prep")
BATCH_FILENAME = "translation_batch_01_of_09.csv"  # Change this for each assigned batch.
INPUT_CSV = DATA_PREP_DIR / "Translation Needed" / BATCH_FILENAME
OUTPUT_CSV = DATA_PREP_DIR / "Translation Done" / BATCH_FILENAME
APP_ID_COLUMN = "AppID"
NAME_COLUMN = "Name"
DESCRIPTION_COLUMN = "About the game"
MODEL_ID = "Qwen/Qwen3-4B"
MAX_NEW_TOKENS = 3000

if not INPUT_CSV.is_file():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}. Check DATA_PREP_DIR and BATCH_FILENAME.")
if INPUT_CSV.resolve() == OUTPUT_CSV.resolve():
    raise ValueError("Input and output paths must be different; the translated file belongs in Translation Done.")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. In Colab, select Runtime > Change runtime type > GPU, then rerun.")
print("Input:", INPUT_CSV)
print("Output:", OUTPUT_CSV)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GiB")

In [ ]:
# Download Qwen3-4B from Hugging Face and load it in 4-bit mode to fit common Colab GPUs.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print(f"Loaded {MODEL_ID}; model input device: {model.get_input_embeddings().weight.device}")

In [ ]:
def atomic_write_csv(frame, path):
    """Write a complete CSV beside its destination, flush it, then replace the destination."""
    path = Path(path)
    temp_path = path.with_name(path.name + ".tmp")
    try:
        with temp_path.open("w", encoding="utf-8-sig", newline="") as handle:
            frame.to_csv(handle, index=False)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_path, path)
    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise


def translate_description(description):
    description = "" if pd.isna(description) else str(description)
    if not description.strip():
        raise ValueError("Encountered an empty description; it was left in the input file.")
    messages = [
        {"role": "system", "content": (
            "You carefully prepare Steam game descriptions for an English-language recommender. "
            "Translate every non-English part into natural English. Preserve existing English, facts, "
            "names, and the original meaning. Do not summarize, add claims, or explain your work. "
            "Remove emojis and HTML markup. Return only the finished description."
        )},
        {"role": "user", "content": description},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=False)
    prompt_length = inputs["input_ids"].shape[1]
    max_context = int(getattr(model.config, "max_position_embeddings", 32768))
    output_budget = min(MAX_NEW_TOKENS, max_context - prompt_length - 8)
    if output_budget < 64:
        raise ValueError(f"Description prompt is too long ({prompt_length} tokens); it remains in the input for manual handling.")
    input_device = model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=output_budget,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(generated[0, prompt_length:], skip_special_tokens=True).strip()
    answer = re.sub(r"<think>.*?</think>", " ", answer, flags=re.IGNORECASE | re.DOTALL)
    answer = re.sub(r"^\s*(?:translation|cleaned description)\s*:\s*", "", answer, flags=re.IGNORECASE)
    answer = re.sub(r"\s+", " ", answer).strip()
    if not answer:
        raise RuntimeError("Qwen returned an empty response; the input row was not removed.")
    return answer


pending = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", low_memory=False, dtype={APP_ID_COLUMN: "string"})
required_columns = {APP_ID_COLUMN, NAME_COLUMN, DESCRIPTION_COLUMN}
missing_columns = sorted(required_columns - set(pending.columns))
if missing_columns:
    raise KeyError(f"Input CSV is missing required columns: {missing_columns}")
if pending[APP_ID_COLUMN].isna().any() or pending[APP_ID_COLUMN].duplicated().any():
    raise ValueError("Input CSV must have a non-missing, unique AppID on every row.")

if OUTPUT_CSV.exists():
    processed = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig", low_memory=False, dtype={APP_ID_COLUMN: "string"})
    if list(processed.columns) != list(pending.columns):
        raise ValueError("Existing output columns differ from the input. Use a matching output file or resolve the schema first.")
    if processed[APP_ID_COLUMN].isna().any() or processed[APP_ID_COLUMN].duplicated().any():
        raise ValueError("Output CSV has missing or duplicate AppIDs; resolve it before resuming.")
else:
    processed = pd.DataFrame(columns=pending.columns)
    atomic_write_csv(processed, OUTPUT_CSV)

# Recover if the runtime stopped after saving output but before removing the same row from input.
already_saved = set(processed[APP_ID_COLUMN].astype(str))
overlap_mask = pending[APP_ID_COLUMN].astype(str).isin(already_saved)
reconciled_count = int(overlap_mask.sum())
if reconciled_count:
    pending = pending.loc[~overlap_mask].copy().reset_index(drop=True)
    atomic_write_csv(pending, INPUT_CSV)
    print(f"Resume recovery: removed {reconciled_count:,} AppIDs already safely present in output.")

print(f"Already translated: {len(processed):,}")
print(f"Still in input queue: {len(pending):,}")

started_at = time.time()
while not pending.empty:
    row_index = pending.index[0]
    row = pending.loc[row_index].copy()
    app_id = str(row[APP_ID_COLUMN])
    game_name = str(row[NAME_COLUMN])
    translated = translate_description(row[DESCRIPTION_COLUMN])
    row[DESCRIPTION_COLUMN] = translated

    # Save the translated row first. Only after that succeeds do we remove the source row.
    processed = pd.concat([processed, row.to_frame().T], ignore_index=True)
    atomic_write_csv(processed, OUTPUT_CSV)
    pending = pending.drop(index=row_index).reset_index(drop=True)
    atomic_write_csv(pending, INPUT_CSV)

    elapsed_minutes = (time.time() - started_at) / 60
    print(f"Translated AppID {app_id} | {game_name} | {len(processed):,} saved | {len(pending):,} remaining | {elapsed_minutes:.1f} min elapsed")

print("Batch complete.")
print("Translated output:", OUTPUT_CSV)
print("Input queue is empty:", len(pd.read_csv(INPUT_CSV, encoding="utf-8-sig")) == 0)